# Notebook 4 — Statistical Analysis
**Input:** `data/processed/gtd_cleaned.csv`  
**Goal:** Apply statistical tests and quantitative analysis to uncover significant patterns and relationships in the data.  
> **Note:** EDA (Notebook 3) covers distributions and visual trends. This notebook focuses on hypothesis testing, effect sizes, and inferential statistics.

### Imports
Loads libraries for statistical testing, effect size calculation, and numerical analysis.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data

### Read Cleaned CSV
Loads the fully cleaned dataset produced by Notebook 2.

In [2]:
df = pd.read_csv('../data/processed/gtd_cleaned.csv')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

Shape: 34,268 rows × 9 columns


,Year,Month,Duration,Country,Region,City,Crime,Reason,Individual
0,1970,1,0,United States,North America,Cairo,1,To protest the Cairo Illinois Police Deparment,0
1,1970,1,0,United States,North America,Madison,1,To protest the War in Vietnam and the draft,0
2,1970,1,0,United States,North America,Madison,1,To protest the War in Vietnam and the draft,0
3,1970,1,0,United States,North America,Denver,1,Protest the draft and Vietnam War,0
4,1970,1,0,United States,North America,Rio Piedras,1,To protest United States owned businesses in P...,0


## 2. Descriptive Statistics

### Central Tendency & Dispersion
Computes mean, median, standard deviation, variance, skewness, and kurtosis for all numeric columns. These go beyond the basic `describe()` in EDA to quantify the shape and spread of each distribution.

In [3]:
numeric_cols = ['Year', 'Month', 'Duration', 'Crime', 'Individual']

stats_df = pd.DataFrame({
    'Mean'     : df[numeric_cols].mean(),
    'Median'   : df[numeric_cols].median(),
    'Std Dev'  : df[numeric_cols].std(),
    'Variance' : df[numeric_cols].var(),
    'Skewness' : df[numeric_cols].skew(),
    'Kurtosis' : df[numeric_cols].kurt(),
}).round(4)

stats_df

,Mean,Median,Std Dev,Variance,Skewness,Kurtosis
Year,2009.6168,2010.0,6.6487,44.2054,-3.3764,15.7589
Month,6.4902,7.0,3.3560,11.2624,-0.0003,-1.1467
Duration,0.0772,0.0,0.2668,0.0712,3.1694,8.0456
Crime,0.9863,1.0,0.1161,0.0135,-8.3811,68.2462
Individual,0.0106,0.0,0.1025,0.0105,9.5479,89.1668


## 3. Attack Frequency Over Time

### Year-over-Year Growth Rate
Calculates the percentage change in attack count from one year to the next. Identifies years with the largest spikes or drops in activity.

In [4]:
yearly = df['Year'].value_counts().sort_index()
yoy = yearly.pct_change().mul(100).round(2)

yoy_df = pd.DataFrame({'Attacks': yearly, 'YoY Change (%)': yoy})
print('Top 10 years with highest YoY growth:')
print(yoy_df.nlargest(10, 'YoY Change (%)'))
print('\nTop 10 years with largest YoY decline:')
print(yoy_df.nsmallest(10, 'YoY Change (%)'))

Top 10 years with highest YoY growth:
      Attacks  YoY Change (%)
Year                         
2008     4042          594.50
1998      158          558.33
1984       46          155.56
2005      420          112.12
1978       35          105.88
1999      308           94.94
1997       24           84.62
1980       20           81.82
2000      513           66.56
1974       31           55.00

Top 10 years with largest YoY decline:
      Attacks  YoY Change (%)
Year                         
1979       11          -68.57
1972       40          -64.60
2012     1828          -57.15
1971      113          -51.08
1973       20          -50.00
1985       26          -43.48
1996       13          -43.48
2003      216          -42.09
1975       18          -41.94
2002      373          -41.26


### Linear Trend Test (Mann-Kendall)
Applies the Mann-Kendall trend test to the yearly attack counts to determine whether there is a statistically significant monotonic trend over time.
- **H₀:** No monotonic trend in attack frequency over time.
- **H₁:** There is a monotonic trend (increasing or decreasing).

In [5]:
tau, p_value = stats.kendalltau(yearly.index, yearly.values)

print(f'Kendall Tau : {tau:.4f}')
print(f'P-value     : {p_value:.6f}')
print()
if p_value < 0.05:
    direction = 'increasing' if tau > 0 else 'decreasing'
    print(f'Result: Statistically significant {direction} trend (p < 0.05).')
else:
    print('Result: No statistically significant trend detected (p ≥ 0.05).')

Kendall Tau : 0.5337
P-value     : 0.000000

Result: Statistically significant increasing trend (p < 0.05).


## 4. Regional Differences

### Attack Count by Region — Proportions & Concentration
Calculates the proportion of total attacks attributed to each region and the cumulative share. Identifies which regions account for the majority of incidents.

In [6]:
region_counts = df['Region'].value_counts()
region_pct    = (region_counts / len(df) * 100).round(2)
region_cum    = region_pct.cumsum().round(2)

region_df = pd.DataFrame({
    'Attacks'       : region_counts,
    'Share (%)'     : region_pct,
    'Cumulative (%)': region_cum
})
region_df

,Attacks,Share (%),Cumulative (%)
Region,,,
South Asia,12491,36.45,36.45
Middle East & North Africa,10103,29.48,65.93
Southeast Asia,3410,9.95,75.88
Sub-Saharan Africa,3066,8.95,84.83
North America,1382,4.03,88.86
Western Europe,1330,3.88,92.74
Eastern Europe,1147,3.35,96.09
South America,1010,2.95,99.04
Central Asia,132,0.39,99.43


### Chi-Square Test — Region vs Duration
Tests whether attack duration (>24 hrs vs ≤24 hrs) is independent of the region where the attack occurred.
- **H₀:** Duration and Region are independent.
- **H₁:** Duration and Region are associated.

In [7]:
contingency = pd.crosstab(df['Region'], df['Duration'])
chi2, p, dof, expected = stats.chi2_contingency(contingency)

print(f'Chi-Square Statistic : {chi2:.4f}')
print(f'Degrees of Freedom   : {dof}')
print(f'P-value              : {p:.6f}')
print()
if p < 0.05:
    print('Result: Significant association between Region and Duration (p < 0.05).')
else:
    print('Result: No significant association between Region and Duration (p ≥ 0.05).')

Chi-Square Statistic : 701.2037
Degrees of Freedom   : 11
P-value              : 0.000000

Result: Significant association between Region and Duration (p < 0.05).


### Chi-Square Test — Region vs Individual
Tests whether individual attacks (lone actor vs group) are distributed differently across regions.
- **H₀:** Individual flag and Region are independent.
- **H₁:** Individual flag and Region are associated.

In [8]:
contingency2 = pd.crosstab(df['Region'], df['Individual'])
chi2_2, p_2, dof_2, _ = stats.chi2_contingency(contingency2)

print(f'Chi-Square Statistic : {chi2_2:.4f}')
print(f'Degrees of Freedom   : {dof_2}')
print(f'P-value              : {p_2:.6f}')
print()
if p_2 < 0.05:
    print('Result: Significant association between Region and Individual attacks (p < 0.05).')
else:
    print('Result: No significant association between Region and Individual attacks (p ≥ 0.05).')

Chi-Square Statistic : 3763.7975
Degrees of Freedom   : 11
P-value              : 0.000000

Result: Significant association between Region and Individual attacks (p < 0.05).


## 5. Temporal Patterns

### Monthly Attack Distribution — Kruskal-Wallis Test
Tests whether the number of attacks differs significantly across months. Uses the non-parametric Kruskal-Wallis test since attack counts are not normally distributed.
- **H₀:** Attack counts are the same across all months.
- **H₁:** At least one month has a significantly different attack count.

In [9]:
monthly_groups = [df[df['Month'] == m]['Year'].count() for m in range(1, 13)]

# Group yearly counts by month across all years
monthly_yearly = [
    df[df['Month'] == m].groupby('Year').size().values
    for m in range(1, 13)
]

h_stat, p_kw = stats.kruskal(*monthly_yearly)

print(f'Kruskal-Wallis H : {h_stat:.4f}')
print(f'P-value          : {p_kw:.6f}')
print()
if p_kw < 0.05:
    print('Result: Significant difference in attack counts across months (p < 0.05).')
else:
    print('Result: No significant difference in attack counts across months (p ≥ 0.05).')

Kruskal-Wallis H : 3.8825
P-value          : 0.973230

Result: No significant difference in attack counts across months (p ≥ 0.05).


### Duration Rate by Year
Calculates the proportion of extended attacks (Duration = 1) per year. Shows whether the rate of prolonged attacks has changed over time.

In [10]:
duration_rate = df.groupby('Year')['Duration'].mean().mul(100).round(2)

print('Extended attack rate (%) per year:')
print(duration_rate.to_string())

Extended attack rate (%) per year:
Year
1970     0.43
1971     0.00
1972     0.00
1973     5.00
1974     0.00
1975     5.56
1976     0.00
1977    17.65
1978     2.86
1979     0.00
1980     0.00
1981     0.00
1982     3.85
1983     0.00
1984     0.00
1985     0.00
1986     2.94
1987     3.85
1988     4.00
1989     0.00
1990     4.55
1991     0.00
1992     0.00
1994     0.00
1995     0.00
1996     7.69
1997     0.00
1998     7.59
1999    12.66
2000     7.02
2001     8.82
2002     5.63
2003     3.24
2004     8.59
2005     7.14
2006     5.39
2007     4.12
2008     5.69
2009     5.86
2010     6.81
2011     6.00
2012     4.65
2013     6.08
2014    13.29
2015    13.15
2016    17.34
2017    11.64


## 6. Binary Feature Relationships

### Chi-Square Test — Duration vs Individual
Tests whether extended attacks are more or less likely to be carried out by individuals vs groups.
- **H₀:** Duration and Individual are independent.
- **H₁:** Duration and Individual are associated.

In [11]:
ct = pd.crosstab(df['Duration'], df['Individual'])
chi2_di, p_di, dof_di, _ = stats.chi2_contingency(ct)

print('Contingency Table (Duration vs Individual):')
print(ct)
print()
print(f'Chi-Square : {chi2_di:.4f}')
print(f'P-value    : {p_di:.6f}')
print()
if p_di < 0.05:
    print('Result: Significant association between Duration and Individual (p < 0.05).')
else:
    print('Result: No significant association between Duration and Individual (p ≥ 0.05).')

Contingency Table (Duration vs Individual):
Individual      0    1
Duration              
0           31264  360
1            2640    4

Chi-Square : 21.6924
P-value    : 0.000003

Result: Significant association between Duration and Individual (p < 0.05).


### Chi-Square Test — Crime vs Individual
Tests whether attacks classified as criminal are more or less likely to be carried out by individuals.
- **H₀:** Crime and Individual are independent.
- **H₁:** Crime and Individual are associated.

In [12]:
ct2 = pd.crosstab(df['Crime'], df['Individual'])
chi2_ci, p_ci, dof_ci, _ = stats.chi2_contingency(ct2)

print('Contingency Table (Crime vs Individual):')
print(ct2)
print()
print(f'Chi-Square : {chi2_ci:.4f}')
print(f'P-value    : {p_ci:.6f}')
print()
if p_ci < 0.05:
    print('Result: Significant association between Crime and Individual (p < 0.05).')
else:
    print('Result: No significant association between Crime and Individual (p ≥ 0.05).')

Contingency Table (Crime vs Individual):
Individual      0    1
Crime                 
0             449   19
1           33455  345

Chi-Square : 37.7287
P-value    : 0.000000

Result: Significant association between Crime and Individual (p < 0.05).


## 7. Top Country Analysis

### Attack Rate per Country — Z-Score Outlier Detection
Computes the Z-score for each country's attack count to identify statistical outliers — countries with attack frequencies significantly above or below the mean.

In [13]:
country_counts = df['Country'].value_counts()
z_scores = stats.zscore(country_counts.values)

country_z = pd.DataFrame({
    'Attacks': country_counts.values,
    'Z-Score': z_scores.round(3)
}, index=country_counts.index)

print('Countries with Z-Score > 2 (statistical outliers):')
print(country_z[country_z['Z-Score'] > 2].sort_values('Z-Score', ascending=False))

Countries with Z-Score > 2 (statistical outliers):
             Attacks  Z-Score
Country                      
Iraq            6320    8.117
India           4363    5.511
Pakistan        4227    5.330
Afghanistan     2234    2.678
Philippines     1839    2.152


## 8. Summary of Statistical Findings

### Key Statistical Results
Prints a concise summary of all hypothesis test outcomes.

In [14]:
results = [
    ('Temporal trend (Kendall Tau)',        p_value,  tau),
    ('Region vs Duration (Chi-Square)',     p,        chi2),
    ('Region vs Individual (Chi-Square)',   p_2,      chi2_2),
    ('Monthly seasonality (Kruskal-Wallis)',p_kw,     h_stat),
    ('Duration vs Individual (Chi-Square)', p_di,     chi2_di),
    ('Crime vs Individual (Chi-Square)',    p_ci,     chi2_ci),
]

print(f'{"Test":<45} {"Statistic":>12} {"P-Value":>12} {"Significant?":>14}')
print('-' * 85)
for name, pval, stat in results:
    sig = 'Yes' if pval < 0.05 else 'No'
    print(f'{name:<45} {stat:>12.4f} {pval:>12.6f} {sig:>14}')

Test                                             Statistic      P-Value   Significant?
-------------------------------------------------------------------------------------
Temporal trend (Kendall Tau)                        0.5337     0.000000            Yes
Region vs Duration (Chi-Square)                   701.2037     0.000000            Yes
Region vs Individual (Chi-Square)                3763.7975     0.000000            Yes
Monthly seasonality (Kruskal-Wallis)                3.8825     0.973230             No
Duration vs Individual (Chi-Square)                21.6924     0.000003            Yes
Crime vs Individual (Chi-Square)                   37.7287     0.000000            Yes
